# Exploratory data analysis

Descriptive analysis of the multilingual training corpus for narrative classification.

Sections 1–8 reconstruct and validate the data independently of other notebook sessions. Sections 9–12 summarize the eligible training articles, label frequencies, article lengths and label cardinality. No classifier is trained and no model-selection parameter is estimated here.

**Inputs:** `training_NLP.zip`, `test_NLP.zip` and `kit_setup_nlp.zip`, archived or extracted. **Execution:** run the notebook in order on CPU. Required dependencies are NumPy, pandas, scikit-learn, Matplotlib and PyTorch for the shared seed helper. No automatic installation or model downloads are performed.

**Outputs:** CSV tables, PNG/PDF figures and `eda_summary.json` under `OUTPUT_ROOT / 'eda'`. Training inputs are read-only. Test texts are used only for the overlap audit; test annotations are not loaded.

The supplied test set was consulted during earlier development and is not an untouched benchmark. The 96-label fine taxonomy is a project definition. These descriptive statistics cover the full eligible training corpus, not individual inner-training folds.

## 1. Input and output paths

Defaults target Kaggle. For local execution, set `NLP_INPUT_ROOT` and `NLP_OUTPUT_ROOT` before running this cell. Optional source overrides are `NLP_TRAIN_SOURCE`, `NLP_TEST_SOURCE` and `NLP_KIT_DIR`.

An extracted training/test source override must point to the directory containing `training/` or `test/`, respectively. A kit override must point directly to the directory containing `nlp_setup.py`. If multiple copies are attached, choose explicit paths.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, shutil

INPUT_ROOT = Path(os.environ.get('NLP_INPUT_ROOT', '/kaggle/input'))
OUTPUT_ROOT = Path(os.environ.get('NLP_OUTPUT_ROOT', '/kaggle/working/nlp_setup'))
TRAIN_OVERRIDE = os.environ.get('NLP_TRAIN_SOURCE') or None
TEST_OVERRIDE = os.environ.get('NLP_TEST_SOURCE') or None
KIT_OVERRIDE = os.environ.get('NLP_KIT_DIR') or None
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT_ROOT)
print('Output:', OUTPUT_ROOT)


## 2. Load the support kit and validate the taxonomy

An extracted kit or a ZIP archive is accepted. An archived kit is extracted into the output directory with path-containment checks. A unique support-module directory is required, including when the archive contains a wrapper directory. The selected module is loaded explicitly to avoid reusing a module from an earlier session import.

In [ ]:
if KIT_OVERRIDE:
    KIT_DIR = Path(KIT_OVERRIDE)
else:
    found = sorted({p.parent for p in INPUT_ROOT.rglob('nlp_setup.py')})
    if len(found) == 1:
        KIT_DIR = found[0]
    elif len(found) > 1:
        raise ValueError('Più kit trovati: impostare KIT_OVERRIDE nella cella 1.')
    else:
        archives = list(INPUT_ROOT.rglob('kit_setup_nlp.zip'))
        if len(archives) != 1:
            raise ValueError('Attach exactly one kit_setup_nlp.zip or set KIT_OVERRIDE.')
        KIT_DIR = OUTPUT_ROOT / '_kit'
        KIT_DIR.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as z:
            for info in z.infolist():
                target = (KIT_DIR / info.filename).resolve()
                if not target.is_relative_to(KIT_DIR.resolve()):
                    raise ValueError('Invalid path in kit archive.')
            z.extractall(KIT_DIR)
        extracted = sorted({p.parent for p in KIT_DIR.rglob('nlp_setup.py')})
        if len(extracted) != 1:
            raise ValueError('Extracted archive must contain exactly one support kit.')
        KIT_DIR = extracted[0]

import importlib.util
if not (KIT_DIR / 'nlp_setup.py').is_file():
    raise FileNotFoundError('KIT_DIR must contain nlp_setup.py.')
spec = importlib.util.spec_from_file_location('nlp_setup', KIT_DIR / 'nlp_setup.py')
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)
import numpy as np
import pandas as pd
from IPython.display import display

taxonomy = json.loads((KIT_DIR / 'tassonomia_progetto_v1.json').read_text(encoding='utf-8'))
setup.require(len(taxonomy['coarse']) == 22 and len(taxonomy['fine']) == 96, 'Unexpected taxonomy.')
setup.require(len(set(taxonomy['coarse'])) == 22, 'Duplicate coarse labels.')
setup.require(len(set(taxonomy['fine'])) == 96, 'Duplicate fine labels.')
setup.require(all(l == 'Other' or l.rsplit(': ', 1)[0] in taxonomy['coarse'] for l in taxonomy['fine']),
              'Missing parent label in taxonomy.')
environment = setup.environment_info()
setup.save_json(OUTPUT_ROOT / 'environment.json', environment)
print('Kit:', KIT_DIR)
print('Classes: 22 coarse / 96 fine. Environment:')
display(environment)


## 3. Load training annotations and test texts

Training labels are validated against the project taxonomy. Repeated labels within a document are deduplicated while retaining a fingerprint of the original annotation row. Unknown labels raise an error.

Test annotations are not read. Training records and test texts must match the previously frozen data fingerprints.

In [ ]:
training_source = setup.locate_source(INPUT_ROOT, 'training_NLP.zip', 'training', TRAIN_OVERRIDE)
test_source = setup.locate_source(INPUT_ROOT, 'test_NLP.zip', 'test', TEST_OVERRIDE)
try:
    train_records = setup.load_training(training_source, taxonomy)
    course_texts = setup.load_course_texts(test_source)
finally:
    for source in (training_source, test_source):
        if source.archive is not None:
            source.archive.close()
expected = json.loads((KIT_DIR / 'expected_data_fingerprints.json').read_text(encoding='utf-8'))
setup.require(setup.fingerprint_records(train_records) == expected['training_records'],
              'Training data do not match the verified version.')
setup.require(setup.fingerprint_records(course_texts) == expected['course_texts'],
              'Test texts do not match the verified version.')

data_counts = pd.DataFrame([
    {'language': lang,
     'training': sum(r['language'] == lang for r in train_records),
     'test_texts_only': sum(r['language'] == lang for r in course_texts)}
    for lang in setup.LANGUAGES
])
display(data_counts)
repeated_docs = sum(r['repeated_labels_removed'] > 0 for r in train_records)
print('Training documents with repeated labels deduplicated:', repeated_docs)


## 4. Duplicate and near-duplicate screening

The lexical audit combines whitespace-normalized exact matches with within-language word-trigram TF-IDF cosine similarity of at least 0.80. Candidate links form connected groups. Training documents connected to supplied test texts are excluded from auxiliary training.

The audit threshold is fixed independently of classifier predictions. Audit TF-IDF features are never reused as classifier inputs. This screening cannot rule out translations or all paraphrases, and candidate similarity is not proof of semantic equivalence.

In [ ]:
similarity = setup.audit_similarity(train_records, course_texts, threshold=0.80)
reference = json.loads((KIT_DIR / 'near_duplicate_audit_reference.json').read_text(encoding='utf-8'))
pair_keys = lambda a: {(p['a'], p['b']) for p in a['pairs']}
setup.require(pair_keys(similarity) == pair_keys(reference),
              'Near-duplicate candidates differ from the reference: check inputs and versions.')
setup.require(similarity['excluded_training_keys'] == reference['excluded_training_keys'],
              'Exclusions differ from the frozen reference.')
setup.save_json(OUTPUT_ROOT / 'near_duplicate_audit.json', similarity)
pd.DataFrame(similarity['pairs']).to_csv(OUTPUT_ROOT / 'similarity_pairs.csv', index=False)
print('Candidate pairs:', len(similarity['pairs']))
print('Excluded auxiliary documents:', len(similarity['excluded_training_keys']))
display(pd.DataFrame({'excluded_document': similarity['excluded_training_keys']}))


## 5. Validate the fixed grouped splits

The version-2 manifest assigns the 399 English articles to 395 groups. In particular, `EN_CC_200107.txt` and `EN_CC_200093.txt`, which differ only in quotation marks, share a group.

Each English article appears in exactly one outer-evaluation partition across the three folds. Within a fold, detected groups must not cross training, validation and outer-evaluation partitions. No documents are removed to balance label frequencies.

In [ ]:
manifest = setup.validate_manifest(train_records, similarity, KIT_DIR / 'split_manifest_v2.csv')
split_counts = manifest.groupby(['fold', 'partition']).size().unstack()
display(split_counts[['inner_train', 'inner_validation', 'outer_evaluation']])
manifest.to_csv(OUTPUT_ROOT / 'verified_split_manifest_v2.csv', index=False)
print('OK: 399 English documents; each assigned to outer evaluation exactly once across three folds.')
print('OK: exact-match and candidate English groups do not cross partitions within a fold.')


## 6. Prepare one fold without training

This example builds the first English-only training partition and its multilabel targets. Inner validation is reserved for model and threshold selection; outer predictions are generated only after those choices are fixed. The partitioning code does not technically prevent access to outer labels, so this separation is a methodological requirement.

A fixed seed improves reproducibility but does not guarantee identical results across software versions and hardware.

In [ ]:
FOLD = 1
TRAIN_SEED = 42
MULTILINGUAL = False
setup.seed_everything(TRAIN_SEED)
fold_data = setup.make_fold(train_records, manifest, similarity, FOLD, MULTILINGUAL)
y_train_coarse = setup.binary_labels([r['coarse'] for r in fold_data['inner_train']], taxonomy['coarse'])
y_train_fine = setup.binary_labels([r['fine'] for r in fold_data['inner_train']], taxonomy['fine'])
print('Fold:', FOLD, '| Multilingual:', MULTILINGUAL)
print({role: len(rows) for role, rows in fold_data.items()})
print('Training targets:', y_train_coarse.shape, y_train_fine.shape)
print('Il seed migliora la riproducibilità, ma non garantisce identità tra hardware e versioni diverse.')


## 7. Verify metric compatibility on synthetic cases

The project metric is checked against the original v3 scorer on shared labels and valid synthetic inputs. The comparison covers mean per-document F1 and its document-level standard deviation; the latter is not a confidence interval.

The original scorer ignores labels outside its vocabulary, whereas the strict project implementation rejects unknown labels. Agreement on shared labels does not make the 96-label project taxonomy equivalent to the official taxonomy.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('official_st2_scorer', KIT_DIR / 'official_v3/subtask2_scorer.py')
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
rng = np.random.default_rng(20261103)
for level, filename in [('coarse', 'subtask2_narratives.txt'), ('fine', 'subtask2_subnarratives.txt')]:
    classes = (KIT_DIR / 'official_v3' / filename).read_text(encoding='utf-8').splitlines()
    gold = [[], [classes[0]], [classes[0]], [classes[0], classes[1]]]
    pred = [[], [classes[0]], [], [classes[1]]]
    for _ in range(30):
        gold.append(rng.choice(classes, size=int(rng.integers(0, 5)), replace=False).tolist())
        pred.append(rng.choice(classes, size=int(rng.integers(0, 5)), replace=False).tolist())
    ours = setup.sample_f1_strict(gold, pred, classes)
    theirs = official.evaluate(gold, pred, classes)
    setup.require(np.allclose([ours['mean'], ours['document_std']], theirs, atol=1e-12),
                  'Scorers disagree on synthetic cases.')
print('OK: metric agrees with v3 on synthetic cases using shared labels.')
try:
    setup.sample_f1_strict([['UNKNOWN_LABEL']], [[]], taxonomy['fine'])
except ValueError:
    print('OK: unknown labels are rejected.')
else:
    raise AssertionError('Unknown label was not rejected.')


## 8. Export the setup summary

`setup_summary.json` records counts, fold sizes, environment and limitations. Preserve this file together with the verified manifest and audit outputs. Subsequent notebooks must reconstruct their required variables in their own sessions; saving this notebook does not transfer its in-memory variables.

No classifier training is performed here. The support kit provides `create_run_directory` for storing later experiments in separate directories.

In [ ]:
fold_sizes = {}
for fold in (1, 2, 3):
    fold_sizes[str(fold)] = {}
    for multi in (False, True):
        parts = setup.make_fold(train_records, manifest, similarity, fold, multi)
        fold_sizes[str(fold)]['multilingual' if multi else 'EN_only'] = {k:len(v) for k,v in parts.items()}
summary = {
    'status': 'SETUP_OK', 'split_version': 'v2', 'taxonomy_version': 'progetto_v1_96fine',
    'training_documents_original': len(train_records),
    'course_test_texts': len(course_texts), 'course_test_labels_loaded': False,
    'coarse_classes': len(taxonomy['coarse']), 'fine_classes': len(taxonomy['fine']),
    'similarity_candidate_pairs': len(similarity['pairs']),
    'auxiliary_documents_excluded': len(similarity['excluded_training_keys']),
    'documents_with_repeated_labels': repeated_docs,
    'fold_sizes': fold_sizes, 'environment': environment,
    'limitations': ['Lexical screen does not rule out translations or all paraphrases.',
                    'The supplied test set was consulted during earlier development; it is not an untouched benchmark.',
                    'Extended taxonomy is a project definition, not an official new release.'],
    'training_started': False
}
setup.save_json(OUTPUT_ROOT / 'setup_summary.json', summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Setup complete. Summary:', OUTPUT_ROOT / 'setup_summary.json')


## 9. Corpus inventory and descriptive statistics

The inventory distinguishes the original corpus from the corpus remaining after auxiliary-document exclusions. Whitespace-normalized text counts describe exact matches, not all near-duplicate groups; retained English duplicates remain grouped in the evaluation splits.

Label cardinality counts distinct labels per article. Length is measured in whitespace-separated units, an approximate descriptive measure that is not interchangeable with model-token counts. Tokenizer limits require a separate tokenization audit.

In [ ]:
from collections import Counter
import matplotlib
import matplotlib.pyplot as plt

EDA_DIR = OUTPUT_ROOT / 'eda'
EDA_DIR.mkdir(parents=True, exist_ok=True)
excluded = set(similarity['excluded_training_keys'])
active_records = [r for r in train_records if r['key'] not in excluded]
en_records = [r for r in active_records if r['language'] == 'EN']
setup.require(len(en_records) == 399 and len(active_records) == 1688, 'Unexpected EDA counts.')

def describe_language(lang):
    selected = [r for r in active_records if r['language'] == lang]
    lengths = np.array([len(r['text'].split()) for r in selected])
    return {
        'language': lang,
        'original_articles': sum(r['language'] == lang for r in train_records),
        'eligible_articles': len(selected),
        'unique_whitespace_normalized_texts': len({r['text_hash'] for r in selected}),
        'mean_coarse_labels': round(float(np.mean([len(r['coarse']) for r in selected])), 2),
        'mean_fine_labels': round(float(np.mean([len(r['fine']) for r in selected])), 2),
        'whitespace_units_median': round(float(np.median(lengths)), 1),
        'whitespace_units_p90': round(float(np.quantile(lengths, .90)), 1),
        'whitespace_units_min': int(lengths.min()), 'whitespace_units_max': int(lengths.max()),
    }

overview = pd.DataFrame([describe_language(lang) for lang in setup.LANGUAGES])
display(overview)
overview.to_csv(EDA_DIR / 'training_overview.csv', index=False)
print('Original training articles:', len(train_records))
print('Eligible training articles after exclusions:', len(active_records))
print('EDA uses training articles only; test annotations are not loaded.')


## 10. Label support and rare categories

Support is the number of training articles containing a label, counted once per article. All taxonomy labels are retained, including labels with zero support. The support-band comparison includes the global `Other` label.

The multilingual column includes English and all eligible auxiliary-language articles. These corpus-level counts do not establish cross-lingual transfer performance and must not be substituted for fold-specific training counts.

In [ ]:
support_tables = {}
for level in ('coarse', 'fine'):
    support = pd.DataFrame({'label': taxonomy[level]})
    for lang in setup.LANGUAGES:
        counts = Counter(label for r in active_records if r['language'] == lang for label in r[level])
        support[lang] = [counts.get(label, 0) for label in support['label']]
    support['multilingual'] = support[setup.LANGUAGES].sum(axis=1)
    support_tables[level] = support
    support.to_csv(EDA_DIR / f'label_support_{level}.csv', index=False)

fine_support = support_tables['fine']
def support_band(n):
    return '0' if n == 0 else '1' if n == 1 else '2–4' if n < 5 else '5–9' if n < 10 else '10+'
bands = ['0', '1', '2–4', '5–9', '10+']
rarity = pd.DataFrame({'articles_per_label': bands})
for scope in ['EN', 'multilingual']:
    counts = Counter(support_band(int(n)) for n in fine_support[scope])
    rarity[scope] = [counts[b] for b in bands]
display(rarity)
rarity.to_csv(EDA_DIR / 'fine_support_bands.csv', index=False)
print('Fine labels absent from the eligible multilingual training corpus:')
display(fine_support.loc[fine_support.multilingual == 0, ['label']])
print('10 categorie coarse più frequenti nel training EN:')
display(support_tables['coarse'].sort_values('EN', ascending=False).head(10)[['label','EN','multilingual']])

for level, table in support_tables.items():
    assert int(table['multilingual'].sum()) == sum(len(r[level]) for r in active_records)
    assert int(table['EN'].sum()) == sum(len(r[level]) for r in en_records)
assert int(rarity['EN'].sum()) == len(taxonomy['fine'])
assert int(rarity['multilingual'].sum()) == len(taxonomy['fine'])


## 11. Distribution plots

Figures are exported as PNG and PDF. English coarse-label prevalence is a percentage of articles; percentages need not sum to 100 because articles may have multiple labels. The length plot reports whitespace-separated units and does not measure tokenizer truncation.

In [ ]:
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
def export_figure(fig, stem):
    fig.savefig(EDA_DIR / (stem + '.png'), dpi=180, bbox_inches='tight')
    fig.savefig(EDA_DIR / (stem + '.pdf'), bbox_inches='tight')
    plt.show()
    plt.close(fig)

coarse_plot = support_tables['coarse'].sort_values('EN', ascending=True)
fig, ax = plt.subplots(figsize=(11, 8))
ax.barh(coarse_plot.label, 100 * coarse_plot.EN / len(en_records), color='#2877a3')
ax.set_xlabel('English training articles containing the label (%)')
ax.set_title('Coarse-label prevalence — English training (n=399)')
ax.grid(axis='x', alpha=.2); ax.set_axisbelow(True)
fig.tight_layout(); export_figure(fig, '01_coarse_prevalence_EN')

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(bands))
ax.bar(x-.18, rarity.EN, width=.36, label='English (399 articles)', color='#2877a3')
ax.bar(x+.18, rarity.multilingual, width=.36, label='Multilingual (1688 articles)', color='#d68a36')
ax.set_xticks(x, bands); ax.set_xlabel('Training articles per fine label')
ax.set_ylabel('Number of fine labels'); ax.set_title('Fine-label support — project taxonomy (96 labels)')
ax.legend(); fig.tight_layout(); export_figure(fig, '02_fine_label_support')

fig, ax = plt.subplots(figsize=(8, 4.5))
lengths_by_language = [[len(r['text'].split()) for r in active_records if r['language'] == lang]
                       for lang in setup.LANGUAGES]
ax.boxplot(lengths_by_language)
ax.set_xticks(np.arange(1,6), setup.LANGUAGES)
ax.set_ylabel('Whitespace-separated units per article')
ax.set_title('Article lengths — usable training data')
fig.tight_layout(); export_figure(fig, '03_training_lengths')

fig, ax = plt.subplots(figsize=(8, 4.5))
for level, offset, color in [('coarse', -.18, '#2877a3'), ('fine', .18, '#d68a36')]:
    counts = Counter(len(r[level]) for r in en_records)
    values = np.arange(1, max(len(r[l]) for r in en_records for l in ('coarse','fine'))+1)
    ax.bar(values+offset, [counts.get(int(v),0) for v in values], width=.36, label=level, color=color)
ax.set_xticks(values); ax.set_xlabel('Distinct labels per article'); ax.set_ylabel('Number of articles')
ax.set_title('Label cardinality — English training (n=399)'); ax.legend()
fig.tight_layout(); export_figure(fig, '04_label_cardinality_EN')


## 12. Export analysis summary

The summary records corpus counts, label support, label cardinality and length statistics. Figures and tables describe the training corpus; they are not classification results. Model thresholds and tokenizer limits are not estimated by this analysis.

In [ ]:
eda_summary = {
    'status': 'EDA_OK', 'split_version': 'v2', 'taxonomy': 'project_v1_96fine',
    'training_original': len(train_records), 'training_usable': len(active_records),
    'EN_articles': len(en_records),
    'EN_fine_labels_observed': int((fine_support.EN > 0).sum()),
    'EN_fine_labels_singleton': int((fine_support.EN == 1).sum()),
    'EN_fine_labels_support_1_to_4': int(((fine_support.EN > 0) & (fine_support.EN < 5)).sum()),
    'multilingual_fine_labels_observed': int((fine_support.multilingual > 0).sum()),
    'EN_mean_coarse_labels': float(np.mean([len(r['coarse']) for r in en_records])),
    'EN_mean_fine_labels': float(np.mean([len(r['fine']) for r in en_records])),
    'EN_whitespace_length_mean': float(np.mean([len(r['text'].split()) for r in en_records])),
    'EN_whitespace_length_median': float(np.median([len(r['text'].split()) for r in en_records])),
    'length_measure': 'whitespace units, not model tokens',
    'matplotlib_version': matplotlib.__version__,
    'course_test_labels_loaded': False, 'training_started': False,
    'figures': sorted(p.name for p in EDA_DIR.glob('*.png')),
}
setup.save_json(EDA_DIR / 'eda_summary.json', eda_summary)
print(json.dumps(eda_summary, ensure_ascii=False, indent=2))
print('Analysis summary:', EDA_DIR / 'eda_summary.json')
